# ARGUS · minimal training demo

> **A polygraph for self-improving AI.**
> ARGUS is the OpenEnv environment that catches an AI lying to itself about whether it's learning.
> Most self-improvement loops trust their own metric. We don't.

This Colab runs the entire ARGUS loop end-to-end on Colab's free T4 in ≈30&nbsp;minutes — same code as
the deployed Hugging Face Space, just imported into the notebook kernel.

**What this notebook proves**
- The env runs end-to-end: `propose → solve → score → SFT → improvement`
- The chain-consensus reward (outcome × process) is the actual training signal
- The metacognition stack — capability map, lie taxonomy, ERCV refusal — fires live during training
- A small open model genuinely improves on GSM8K-style word problems

**Runtime**: ≈8 min on Colab free T4 (Qwen2.5-1.5B in 4-bit · 1 episode × 16 steps · eval n=10).
The headline 15-episode result on the laptop uses much larger settings — see ARCHITECTURE.md.
**To start**: `Runtime` → `Change runtime type` → `T4 GPU`, then `Runtime` → `Run all`.

| resource | link |
| --- | --- |
| Live demo (hosted env) | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space](https://vaibhav-pandeyy-argus-self-learning-env.hf.space) |
| HF Space repo | [https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV](https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV) |
| GitHub mirror | [https://github.com/VP-Vaibhav-Pandey/argus-env](https://github.com/VP-Vaibhav-Pandey/argus-env) |

---


## 1 · Install dependencies

≈90 s on a cold Colab session.

In [ ]:
%%capture
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets huggingface_hub
!pip install -q sentencepiece numpy scipy scikit-learn


## 2 · Clone the env

The deployed HF Space is itself a git repository. We clone it directly — same code that serves the live demo at <https://vaibhav-pandeyy-argus-self-learning-env.hf.space>.

In [ ]:
import os
REPO_DIR = 'argus'
HF_SPACE_GIT = 'https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV'

if not os.path.exists(REPO_DIR):
    !git clone -q {HF_SPACE_GIT} {REPO_DIR}
%cd {REPO_DIR}
print('repo contents:')
!ls


## 3 · GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU'
print(f'GPU:   {torch.cuda.get_device_name(0)}')
print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'torch: {torch.__version__}')


## 4 · Load model + LoRA

Qwen2.5-1.5B-Instruct in 4-bit. Same architecture family as the Qwen3-1.7B used in the headline run,
just smaller and universally available so the Colab demo cold-starts in seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


## 5 · Boot the ARGUS env in-process

Same env code as the deployed Space, just imported here. No HTTP round-trips — fastest path for a Colab
demo. (See the optional cell at the end for hitting the deployed Space over HTTP instead.)

In [ ]:
import sys
sys.path.insert(0, '.')
from src.env.self_improvement_env import make_env, chain_consensus_score

env = make_env(
    seed=0,
    solver_k=4,            # 4 chains per problem (Colab demo · headline run uses 14)
    frontier_low=0.4,
    frontier_high=0.8,
    retest_every=4,
    memory_size=64,
)
obs, info = env.reset(seed=0)
print(f'env initialised · mode={obs["mode"]}  skill={obs["skill_level"]:.3f}')


## 6 · Pre-eval baseline · pass@1 on GSM8K (n=20)

In [ ]:
import re
from datasets import load_dataset

EVAL_N = 10
gsm = load_dataset('gsm8k', 'main', split='test').shuffle(seed=42).select(range(EVAL_N))

def extract_answer(text):
    m = re.search(r'####\s*(-?\d+(?:[\.,]\d+)?)', text or '')
    if m: return m.group(1).replace(',', '')
    nums = re.findall(r'-?\d+(?:\.\d+)?', text or '')
    return nums[-1] if nums else ''

def gold(answer_field):
    m = re.search(r'####\s*(-?\d+(?:[\.,]\d+)?)', answer_field)
    return m.group(1).replace(',', '') if m else ''

def generate(prompt, max_new=192, n=1, temperature=0.8):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(text, return_tensors='pt').to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new,
        num_return_sequences=n,
        pad_token_id=tokenizer.pad_token_id,
    )
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        gen_kwargs.update(do_sample=False)  # greedy
    out = model.generate(**inp, **gen_kwargs)
    return [tokenizer.decode(o[inp['input_ids'].shape[1]:], skip_special_tokens=True) for o in out]

def eval_passat1():
    correct = 0
    for ex in gsm:
        q = ex['question']
        prompt = q + "\n\nSolve step by step. End your answer with '#### N' where N is the final number."
        chain = generate(prompt, n=1, temperature=0.0, max_new=192)[0]
        if extract_answer(chain) == gold(ex['answer']):
            correct += 1
    return correct / EVAL_N

pre = eval_passat1()
print(f'pre · pass@1 (n={EVAL_N}) = {pre:.3f}')


## 7 · Self-play loop · the heart of ARGUS

The env alternates `propose` and `solve` modes. We collect every `(problem, chain, weight)` triple that
falls in the **frontier zone** (chain-consensus between 0.4 and 0.8 — neither too easy nor too hard).
Those weighted triples become the SFT signal.

In [ ]:
from src.eval.lie_taxonomy import classify_episode
import time

EPISODES = 1
STEPS_PER_EP = 16
K_CHAINS = 3

training_triples = []   # (problem_text, chain_text, weight)
episode_summaries = []

for ep in range(EPISODES):
    obs, _ = env.reset(seed=ep)
    step_rewards, frontier_steps = [], 0
    t0 = time.time()
    for step in range(STEPS_PER_EP):
        if obs['mode'] == 'propose':
            seed_text = obs.get('context_seed') or ''
            prompt = (
                f"Generate a NEW grade-school math word problem (~{obs.get('frontier_hint', 3)} "
                f"reasoning steps).\n\nReference: {seed_text[:200]}\n\nOutput only the problem."
                if seed_text else
                f"Generate a grade-school math word problem (~{obs.get('frontier_hint', 3)} steps). "
                f"Output only the problem."
            )
            problem = generate(prompt, max_new=96, n=1, temperature=1.0)[0].strip()
            obs, r, term, trunc, info = env.step(problem)
        else:  # solve
            problem_text = obs['problem']  # capture BEFORE step (mode switches)
            chains = generate(problem_text, max_new=96, n=K_CHAINS, temperature=0.8)
            obs, r, term, trunc, info = env.step(chains)
            step_rewards.append(float(r))
            zone = info.get('zone')
            if zone == 'frontier':
                frontier_steps += 1
                # Pick the best chain (highest per-chain agreement)
                cs = chain_consensus_score(chains)
                best_idx = cs.get('best_chain_idx', 0) or 0
                training_triples.append((problem_text, chains[best_idx], float(r)))
        if term or trunc:
            break
    summary = {'ep': ep+1, 'mean_reward': sum(step_rewards)/max(1,len(step_rewards)),
               'frontier_steps': frontier_steps, 'pairs': len(training_triples)}
    episode_summaries.append(summary)
    dt = time.time() - t0
    print(f'ep {ep+1}/{EPISODES} · mean_reward={summary["mean_reward"]:.3f} · '
          f'frontier={frontier_steps} · pairs={len(training_triples)} · {dt:.0f}s')

print(f'\ntraining buffer: {len(training_triples)} weighted (problem, chain) triples')


## 8 · Train with TRL `SFTTrainer` · consensus-weighted SFT

Standard TRL SFT, with one twist: the per-sample loss is multiplied by its chain-consensus weight.
This is **advantage-weighted regression** — high-reward chains contribute more to the gradient than
low-reward ones. The `WeightedSFTTrainer` subclass implements this in `compute_loss`.

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import torch.nn.functional as F

def format_pair(problem, chain):
    msgs = [
        {'role': 'user', 'content': problem},
        {'role': 'assistant', 'content': chain},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

rows = [{'text': format_pair(p, c), 'weight': max(0.1, w)} for p, c, w in training_triples]
if not rows:
    raise RuntimeError('No frontier-zone pairs collected. Increase EPISODES or check env.')
ds = Dataset.from_list(rows)
print(f'training pairs: {len(ds)} · weight range: '
      f'{min(r["weight"] for r in rows):.2f} .. {max(r["weight"] for r in rows):.2f}')

def weighted_collator(batch):
    weights = torch.tensor([float(b['weight']) for b in batch], dtype=torch.float32)
    enc = tokenizer(
        [b['text'] for b in batch],
        padding=True, truncation=True, max_length=512, return_tensors='pt',
    )
    enc['labels'] = enc['input_ids'].clone()
    enc['labels'][enc['attention_mask'] == 0] = -100
    enc['weights'] = weights
    return enc

class WeightedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        weights = inputs.pop('weights')
        labels = inputs['labels']
        out = model(**inputs)
        logits = out.logits[:, :-1].contiguous()
        targets = labels[:, 1:].contiguous()
        nll = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            reduction='none', ignore_index=-100,
        ).view(targets.shape[0], -1)
        mask = (targets != -100).float()
        per_sample = (nll * mask).sum(1) / mask.sum(1).clamp(min=1)
        w = weights.to(per_sample.device, dtype=per_sample.dtype)
        loss = (per_sample * w).sum() / w.sum().clamp(min=1e-6)
        return (loss, out) if return_outputs else loss

cfg = SFTConfig(
    output_dir='argus_run',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    bf16=True,
    report_to='none',
    remove_unused_columns=False,
    # Note: truncation to 512 tokens is handled by our weighted_collator above.
    # Newer TRL versions removed `max_seq_length` from SFTConfig.
)
trainer = WeightedSFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds,
    data_collator=weighted_collator,
)
trainer.train()


## 9 · Post-eval · pass@1 after ARGUS-gated training

In [ ]:
post = eval_passat1()
delta = post - pre

print()
print('  ' + '─'*58)
print(f'  ARGUS minimal training · pass@1 result on n={EVAL_N}')
print('  ' + '─'*58)
print(f'    pre   :  {pre:.3f}')
print(f'    post  :  {post:.3f}')
print(f'    delta :  {delta:+.3f}')
print('  ' + '─'*58)
print()
print(f'  Note: n={EVAL_N} is a small sample for a Colab demo.')
print('  Headline 15-episode run on the laptop uses n=100 with tighter Wilson CIs')
print('  and reports +7.0 pp pass@1 with 7 lies caught and 2 ERCV refusals.')
print(f'  Live demo + full architecture: https://vaibhav-pandeyy-argus-self-learning-env.hf.space')


## 10 · Optional · drive the deployed Space over HTTP

The env you just trained against is the same env serving the public demo. Uncomment to drive it
remotely with the OpenEnv contract. Useful for proving the deployed Space is real and reachable.

In [ ]:
# import requests
# SPACE = 'https://vaibhav-pandeyy-argus-self-learning-env.hf.space'
#
# r = requests.post(f'{SPACE}/reset', json={'seed': 0}); r.raise_for_status()
# session_id = r.json()['session_id']
# print('session:', session_id)
#
# r = requests.post(f'{SPACE}/step', json={
#     'session_id': session_id,
#     'action': 'A bakery sells cupcakes for $3 and brownies for $2. '
#               'On Monday, 24 cupcakes and 18 brownies were sold. '
#               'How much money did the bakery make on Monday?'
# })
# print(r.json())


## Summary

You just ran the ARGUS self-improvement loop end-to-end:

1. **Loaded** Qwen2.5-1.5B-Instruct + LoRA in 4-bit
2. **Connected** to ARGUS in-process (same code as the deployed Space)
3. **Pre-eval** GSM8K pass@1 baseline
4. **Self-play** for 2 episodes — env scored every chain via chain-consensus and tagged frontier ones
5. **SFT** with per-sample-weighted loss using TRL's `SFTTrainer` (subclassed)
6. **Post-eval** to measure delta

**The core idea** — ARGUS scores reasoning chains using **outcome × process consensus**, refuses any
training step where the local "did I learn" signal disagrees with empirical retest (ERCV), and explains
which concept regressed via causal attribution. That is the contribution.

| read more | link |
| --- | --- |
| Live interactive demo | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space](https://vaibhav-pandeyy-argus-self-learning-env.hf.space) |
| OpenEnv API docs | [https://vaibhav-pandeyy-argus-self-learning-env.hf.space/docs](https://vaibhav-pandeyy-argus-self-learning-env.hf.space/docs) |
| HF Space (source) | [https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV](https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV) |
| GitHub mirror | [https://github.com/VP-Vaibhav-Pandey/argus-env](https://github.com/VP-Vaibhav-Pandey/argus-env) |
| Architecture · 3-loop control + lie taxonomy | [ARCHITECTURE.md](https://huggingface.co/spaces/Vaibhav-Pandeyy/Argus-Self-Learning-ENV/blob/main/ARCHITECTURE.md) |
